In [8]:
!pip install jsonpickle

In [6]:
import pandas as pd
import numpy as np
from itertools import product
from run_backtest import load_trader
from backtester import HistoricalData, ReplayBacktester
from strategies.strategy import Trader
from backtester.reporting import summarize_result

In [3]:
P_D1 = "../data/prices_round_0_day_-1.csv"
P_D2 = "../data/prices_round_0_day_-2.csv"
T_D1 = "../data/trades_round_0_day_-1.csv"
T_D2 = "../data/trades_round_0_day_-2.csv"
STEPS = 10000

In [4]:
# ── Grid Search ──────────────────────────────────────────────────────────────
# Parameters to search:
#   er_lam   – EWMA weight on previous mid for Emeralds  (higher = slower)
#   tom_lam  – EWMA weight on previous mid for Tomatoes
#   er_alpha – inventory skew strength for Emeralds
#   tom_alpha– inventory skew strength for Tomatoes

er_lam_values = [0.75, 0.8, 0.85]
tom_lam_values = [0.75, 0.8, 0.85]
er_alpha_values = [0.075, 0.1, 0.125]
tom_alpha_values = [0.04, 0.05, 0.06]

records = []

total = (
    len(er_lam_values)
    * len(tom_lam_values)
    * len(er_alpha_values)
    * len(tom_alpha_values)
)
done = 0

for er_lam in er_lam_values:
    for tom_lam in tom_lam_values:
        for er_alpha in er_alpha_values:
            for tom_alpha in tom_alpha_values:
                trader = Trader(
                    lam=[er_lam, tom_lam],
                    alpha=[er_alpha, tom_alpha],
                )
                data = HistoricalData(P_D1, T_D1, max_rows=STEPS)
                engine = ReplayBacktester(data, trader)
                result = engine.run()

                records.append(
                    {
                        "er_lam": er_lam,
                        "tom_lam": tom_lam,
                        "er_alpha": er_alpha,
                        "tom_alpha": tom_alpha,
                        "final_pnl": result.final_pnl,
                    }
                )

                done += 1
                if done % 10 == 0:
                    print(f"{done}/{total} done...")

results_df = pd.DataFrame(records).sort_values("final_pnl", ascending=False)
print(f"\nTop 5 parameter combinations:")
print(results_df.head())

10/81 done...
20/81 done...
30/81 done...
40/81 done...
50/81 done...
60/81 done...
70/81 done...
80/81 done...

Top 5 parameter combinations:
    er_lam  tom_lam  er_alpha  tom_alpha  final_pnl
31    0.80     0.75     0.100       0.05    16067.0
58    0.85     0.75     0.100       0.05    16067.0
4     0.75     0.75     0.100       0.05    16063.0
1     0.75     0.75     0.075       0.05    16044.0
55    0.85     0.75     0.075       0.05    16044.0
